In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import RidgeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import VotingClassifier

In [2]:
combined = load_npz(r"train.ft.txt\tfidfPlusFeatures.npz").tocsr()
columns = joblib.load(r"train.ft.txt\tfidfPlusFeaturesColumns.pkl")

rating_index = columns.index("Rating")
y = combined[:, rating_index].toarray().ravel()
X = combined[:, [i for i in range(combined.shape[1]) if i != rating_index]]

print("X shape:", X.shape)
print("y shape:", y.shape)

test_sparse = load_npz(r"test.ft.txt\TEST_tfidfPlusFeatures.npz").tocsr()
test_columns = joblib.load(r"test.ft.txt\TEST_tfidfPlusFeaturesColumns.pkl")

rating_idx = test_columns.index("Rating")

y_test = test_sparse[:, rating_idx].toarray().ravel()
X_test = test_sparse[:, [i for i in range(test_sparse.shape[1]) if i != rating_idx]]


X shape: (3586507, 10003)
y shape: (3586507,)


<h2>Logistic Regression</h2>

In [11]:
model = LogisticRegression(
    penalty='l2',        # L2 regularization
    solver='saga',       # supports sparse matrices and L2
    max_iter=3000,       # increased (needed for convergence)
    n_jobs=-1,           # use all CPU cores
    verbose=1
)

model.fit(X, y)

print("Model training completed with L2 regularization.")

joblib.dump(model, "logistic_regression_l2_model.joblib")
print("\nModel saved to 'logistic_regression_l2_model.joblib'")

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


max_iter reached after 14064 seconds
Model training completed with L2 regularization.

Model saved to 'logistic_regression_l2_model.joblib'


In [ ]:
model = joblib.load("logistic_regression_l2_model.joblib")

y_pred_test = model.predict(X_test)

# Evaluation on test data
acc_test = accuracy_score(y_test, y_pred_test)
print(f"\nTest Accuracy: {acc_test:.4f}")

print("\nClassification Report (Test Data):")
print(classification_report(y_test, y_pred_test))

cm_test = confusion_matrix(y_test, y_pred_test)
print("\nConfusion Matrix (Test Data):")
print(cm_test)



Test Accuracy: 0.8853

Classification Report (Test Data):
              precision    recall  f1-score   support

         1.0       0.89      0.89      0.89    199417
         2.0       0.89      0.89      0.89    199056

    accuracy                           0.89    398473
   macro avg       0.89      0.89      0.89    398473
weighted avg       0.89      0.89      0.89    398473


Confusion Matrix (Test Data):
[[176546  22871]
 [ 22848 176208]]


<h2>Linear Support Vector Classifier</h2>

In [6]:
model = LinearSVC(
    penalty='l2',
    loss='squared_hinge',
    dual=False,
    max_iter=5000,
    tol=1e-6,
    C=1.0,
    verbose=10
)

model.fit(X, y)

print("Model training completed with LinearSVC and L2 regularization.")

# Save the model
joblib.dump(model, "linear_svc_l2_model.joblib")
print("\nModel saved to 'linear_svc_l2_model.joblib'")

# Load and prepare test data
test_sparse = load_npz(r"test.ft.txt\TEST_tfidfPlusFeatures.npz").tocsr()
test_columns = joblib.load(r"test.ft.txt\TEST_tfidfPlusFeaturesColumns.pkl")

rating_idx = test_columns.index("Rating")

y_test = test_sparse[:, rating_idx].toarray().ravel()
X_test = test_sparse[:, [i for i in range(test_sparse.shape[1]) if i != rating_idx]]

# Load the trained model
model = joblib.load("linear_svc_l2_model.joblib")

# Predict
y_pred_test = model.predict(X_test)

# Evaluate
acc_test = accuracy_score(y_test, y_pred_test)
print(f"\nTest Accuracy: {acc_test:.4f}")

print("\nClassification Report (Test Data):")
print(classification_report(y_test, y_pred_test))

cm_test = confusion_matrix(y_test, y_pred_test)
print("\nConfusion Matrix (Test Data):")
print(cm_test)


[LibLinear]Model training completed with LinearSVC and L2 regularization.

Model saved to 'linear_svc_l2_model.joblib'

Test Accuracy: 0.8978

Classification Report (Test Data):
              precision    recall  f1-score   support

         1.0       0.90      0.90      0.90    199417
         2.0       0.90      0.90      0.90    199056

    accuracy                           0.90    398473
   macro avg       0.90      0.90      0.90    398473
weighted avg       0.90      0.90      0.90    398473


Confusion Matrix (Test Data):
[[178703  20714]
 [ 20000 179056]]


<h2>Random Forest Classifier</h2>

In [ ]:
svd = TruncatedSVD(n_components=200, random_state=42)

X_reduced = svd.fit_transform(X)
X_test_reduced = svd.transform(X_test)

joblib.dump(svd, "svd_transformer.joblib")
print("TruncatedSVD transformer saved.")

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=30,
    random_state=116,
    n_jobs=-1
)
model.fit(X_reduced, y)

joblib.dump(model, "random_forest_model.joblib") # I deleted it because the model’s size was too large (9GB).
print("Random Forest model saved.")

y_pred_test = model.predict(X_test_reduced)

print("Test Accuracy:", accuracy_score(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))
print(confusion_matrix(y_test, y_pred_test))


TruncatedSVD transformer saved.
Random Forest model saved.
Test Accuracy: 0.8269167547111097
              precision    recall  f1-score   support

         1.0       0.82      0.83      0.83    199417
         2.0       0.83      0.82      0.83    199056

    accuracy                           0.83    398473
   macro avg       0.83      0.83      0.83    398473
weighted avg       0.83      0.83      0.83    398473

[[165708  33709]
 [ 35260 163796]]


<h2>Ridge Classifier</h2>

In [4]:
ridge = RidgeClassifier(alpha=1.0, solver='sag', fit_intercept=False, max_iter=5000)
ridge.fit(X, y)

# Save the model
joblib.dump(ridge, "best_ridge_classifier_model.joblib", compress=3)
print("Ridge Classifier model saved.")

# Predict and evaluate
y_pred_best = ridge.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Classification Report:\n", classification_report(y_test, y_pred_best))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_best))

Ridge Classifier model saved.
Accuracy: 0.891084213986895
Classification Report:
               precision    recall  f1-score   support

         1.0       0.89      0.89      0.89    199417
         2.0       0.89      0.89      0.89    199056

    accuracy                           0.89    398473
   macro avg       0.89      0.89      0.89    398473
weighted avg       0.89      0.89      0.89    398473

Confusion Matrix:
 [[177477  21940]
 [ 21460 177596]]


<h2>Multinomial Naive Bayes</h2>

In [ ]:
# Parameter grid
alphas = [0, 0.5, 1.0]
fit_priors = [True, False]

best_accuracy = 0
best_model = None
best_params = {}

for alpha in alphas:
    for fit_prior in fit_priors:
        print(f"\nTrying alpha={alpha}, fit_prior={fit_prior}")
        nb = MultinomialNB(alpha=alpha, fit_prior=fit_prior)
        nb.fit(X, y)
        y_pred = nb.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        print("Accuracy:", acc)
        
        if acc > best_accuracy:
            best_accuracy = acc
            best_model = nb
            best_params = {"alpha": alpha, "fit_prior": fit_prior}
            best_y_pred = y_pred

# Save best model
joblib.dump(best_model, "best_multinomial_nb_model.joblib", compress=3)

# Show best results
print("\n=== Best MultinomialNB Model ===")
print("Best Parameters:", best_params)
print("Best Accuracy:", best_accuracy)
print("Classification Report:\n", classification_report(y_test, best_y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, best_y_pred))



Trying alpha=0, fit_prior=True
Accuracy: 0.839246322837432

Trying alpha=0, fit_prior=False
Accuracy: 0.8391860929096827

Trying alpha=0.5, fit_prior=True
Accuracy: 0.8392588707390463

Trying alpha=0.5, fit_prior=False
Accuracy: 0.8392287557751718

Trying alpha=1.0, fit_prior=True
Accuracy: 0.8392990240242124

Trying alpha=1.0, fit_prior=False
Accuracy: 0.839263889899692

=== Best MultinomialNB Model ===
Best Parameters: {'alpha': 1.0, 'fit_prior': True}
Best Accuracy: 0.8392990240242124
Classification Report:
               precision    recall  f1-score   support

         1.0       0.83      0.85      0.84    199417
         2.0       0.84      0.83      0.84    199056

    accuracy                           0.84    398473
   macro avg       0.84      0.84      0.84    398473
weighted avg       0.84      0.84      0.84    398473

Confusion Matrix:
 [[168798  30619]
 [ 33416 165640]]


<h2>Ensemble Voting Classifier: Combining The 5 Models</h2>

In [5]:
lr_model = joblib.load("logistic_regression_l2_model.joblib")
svc_model = joblib.load("linear_svc_l2_model.joblib")
rf_model = joblib.load("random_forest_model.joblib")
ridge_model = joblib.load("best_ridge_classifier_model.joblib")
nb_model = joblib.load("best_multinomial_nb_model.joblib")

# SVD was applied for Random Forest, so reduce test set for it
svd = joblib.load("svd_transformer.joblib")
X_reduced = svd.transform(X)
X_test_reduced = svd.transform(X_test)

# Define ensemble with a workaround for the reduced Random Forest input
class CustomVotingClassifier:
    def __init__(self, models):
        self.models = models

    def predict(self, X_full, X_rf):
        from scipy.stats import mode
        predictions = []
        for name, model in self.models:
            if name == 'random_forest':
                y_pred = model.predict(X_rf)
            else:
                y_pred = model.predict(X_full)
            predictions.append(y_pred)
        return mode(predictions, axis=0).mode.ravel()

# Instantiate with all models
voting_clf = CustomVotingClassifier([
    ('logreg', lr_model),
    ('svc', svc_model),
    ('random_forest', rf_model),
    ('ridge', ridge_model),
    ('nb', nb_model)
])

# Get predictions
y_pred_voting = voting_clf.predict(X_test, X_test_reduced)

# Evaluate
print("Voting Classifier Test Accuracy:", accuracy_score(y_test, y_pred_voting))
print("Classification Report:\n", classification_report(y_test, y_pred_voting))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_voting))


Voting Classifier Test Accuracy: 0.8928509585342044
Classification Report:
               precision    recall  f1-score   support

         1.0       0.89      0.89      0.89    199417
         2.0       0.89      0.89      0.89    199056

    accuracy                           0.89    398473
   macro avg       0.89      0.89      0.89    398473
weighted avg       0.89      0.89      0.89    398473

Confusion Matrix:
 [[178113  21304]
 [ 21392 177664]]


<h5>Fares Alfares</h5>